# Table performance of 10 best random forest models

In [ ]:
# Libraries
import numpy as np
import pandas as pd
import dask.dataframe as dd
import sklearn.metrics as metrics

In [ ]:
# Directories
dir02 = '../paper_ecosystem_carbon/output/02_database_prep/'
dir03 = '../paper_ecosystem_carbon/output/03_rf_predict/'
dir08 = '../paper_ecosystem_carbon/output/08_eval/'

---

In [ ]:
# Initialize dask
from dask_jobqueue import SLURMCluster
cluster = SLURMCluster(queue='interactive',
                       cores=24,
                       memory='100 GB',
                       account='bm0891', 
                       interface="ib0",
                       walltime='02:00:00',
                       local_directory='../dask/',
                       job_extra_directives=[
                           '-o ../dask/LOG_worker_%j.o',
                           '-e ../dask/LOG_worker_%j.o'])

# Scale dask clusters
cluster.scale(jobs=10)

# Configurate dashboard url
import dask
dask.config.config.get('distributed').get('dashboard') \
    .update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})

# Create client
from dask.distributed import Client
client = Client(cluster)
client

---

In [ ]:
# Dictionary with ecosystem types and original names
dict_etype = dict(
    trop_ef = ['Tropical/subtropical evergreen forest', 
               'Tropical/subtropical\nevergreen forest'], 
    trop_df = ['Tropical/subtropical deciduous forest', 
               'Tropical/subtropical\ndeciduous forest'],
    temp_ebf = ['Temperate/boreal evergreen broadleaf forest', 
                'Temperate/boreal\nevergreen broadleaf forest'],
    temp_dbf = ['Temperate/boreal deciduous broadleaf forest', 
                'Temperate/boreal\ndeciduous broadleaf forest'],
    temp_enf = ['Temperate/boreal evergreen needleleaf forest', 
                'Temperate/boreal\nevergreen needleleaf forest'],
    temp_dnf = ['Temperate/boreal deciduous needleleaf forest', 
                'Temperate/boreal\ndeciduous needleleaf forest'],
    glob_shrub = ['Shrublands', 'Shrublands'],
    glob_grass = ['Grasslands','Grasslands'],
    glob_tundr = ['Tundra', 'Tundra'],
    glob_mount = ['Montane', 'Montane'])

In [ ]:
# Get database
dd_base = dd.read_parquet(f'{dir02}dd_base_r001.parquet') \
    .repartition(partition_size='1000 MiB') \
    .reset_index(True) \
    .persist()

In [ ]:
def get_df_test(ctype, etype):

    """Returns dataframe with testing data"""
    
    # Get predicted values
    dd_pred = dd.read_parquet(
        f'{dir03}files_predicted_r001/df_rfpred_{ctype}_{etype}.parquet') \
        .repartition(partition_size='1000 MiB') \
        .reset_index(True) \
        .persist()
    
    # Define training/testing carbon (ensure consistency with 02_05_prep_database_r001)
    if ctype == 'cveg':
        if etype in ['trop_ef', 'trop_df', 'temp_ebf', 'temp_dbf', 'temp_enf',
                     'temp_dnf', 'glob_shrub']:
            cbase = 'cveg_biom'    
        if etype in ['glob_tundr', 'glob_mount']:
            cbase = 'cveg_tund'
        if etype in ['glob_grass', 'glob_past']:
            cbase = 'cveg_gras'
        if etype == 'glob_crop':
            cbase = 'cveg_crop'
    if ctype == 'csoil':
            cbase = 'csoil_000_100'
    
    # Prepare dataframe with testing values
    dd_base_sel = dd_base[((dd_base[f'test_{ctype}_{etype}'] == True))] \
        [['lat', 'lon', cbase]].rename(columns={cbase: 'cbase'})

    # Create dataframe with testing values (orig and predicted)
    df_test = dd.merge(dd_base_sel, dd_pred, how='left', on=['lat', 'lon']) \
            .persist()

    return df_test


def get_df_stats(ctype, etype):

    """Returns dataframe with statistics (median, r2, rmse, rrmse, mae, ...)"""
    
    df_test = get_df_test(ctype, etype).compute()
    
    df_stats = pd.DataFrame(columns=['rfr', 'ntest', 'median_test', 
                                     'r2', 'rmse', 'rrmse','mae', 'rmae',
                                     'mbe', 'rmbe'])
    
    for i in range(1,11):
        rfrx = f'rfr_{i}'
        
        # Calculate metrics on the full dataset (using Dask for larger data)
        x_full, y_full = df_test['cbase'], df_test[rfrx]
        
        n_test = len(df_test) # Number of testing data
        v_median_test = np.median(x_full)
        v_r2 = metrics.r2_score(x_full, y_full) # Coefficient of determination
        v_rmse = metrics.root_mean_squared_error(x_full, y_full) # Root-mean-square error
        v_rrmse = (100 * v_rmse / np.median(x_full)) # Relative root-mean-square error
        v_mae = metrics.mean_absolute_error(x_full, y_full) # Mean absolute error
        v_rmae = (100 * v_mae / np.median(x_full)) # Relative mean absolute error
        v_mbe = np.mean(y_full - x_full) # Mean bias error
        v_rmbe = (100 * v_mbe / np.median(x_full)) # Relative mean bias error
        
        df_stats.loc[len(df_stats)] = [rfrx, n_test, v_median_test,
                                       v_r2, v_rmse, v_rrmse, v_mae, v_rmae,
                                       v_mbe, v_rmbe]

    return df_stats


def get_df_stats_sum(ctype):

    """Creates and exports dataframe with statistics summary"""
    
    metric_cols = ['r2', 'rmse', 'rrmse', 'mae', 'rmae']
    rows = []

    for etype, etype_info in dict_etype.items():
        df_stats = get_df_stats(ctype, etype)
        stats = df_stats[metric_cols].agg(['mean', 'min', 'max'])

        row = {'etype': etype_info[0],
               'ntest': int(np.mean(df_stats.ntest)),
               'median_test': int(np.mean(df_stats.median_test))}

        for metric in metric_cols:
            v_mean = stats.loc['mean', metric]
            v_min = stats.loc['min', metric]
            v_max = stats.loc['max', metric]

            if metric in ['r2']:
                row[metric] = f'{round(v_mean, 2):.2f} ({round(v_min, 2):.2f} -- {round(v_max, 2):.2f})'
            else:
                row[metric] = f'{round(v_mean, 1):.1f} ({round(v_min, 1):.1f} -- {round(v_max, 1):.1f})'

        rows.append(row)

    df_out = pd.DataFrame(rows, columns=['etype', 'ntest', 'median_test', *metric_cols])

    # Export
    df_out.to_csv(f'{dir08}tables/table_best10rf_stats_{ctype}.csv', 
               mode='w', index=True)
    
    return df_out

In [ ]:
# Vegetation
get_df_stats_sum('cveg')

In [ ]:
# Soil
get_df_stats_sum('csoil')

---

In [ ]:
cluster.close()